## Containers vs. virtual machines

The cleanest one-line definition of a container: **an isolated process on a host, running its own filesystem and seeing its own network, while sharing the host's kernel.** That sentence does a lot of work — let's pick it apart, then contrast it with a virtual machine.

**An isolated process.** When you start a container, the kernel gives it its own **PID namespace**. Inside, `ps` shows only that container's processes — PID 1 is *its* PID 1, not the host's. From the host those same processes are still visible with different PIDs; the kernel is simply lying to the container about who else is on the box.

**Its own filesystem.** The container sees a root filesystem (`/`) assembled from an **image**. Its `/etc`, `/usr`, `/var` are *not* your host's — they live in a separate area of disk and are mounted in at start.

**Its own network.** By default it gets a virtual interface with its own IP on a Docker-managed bridge. The sockets it opens are *its* sockets.

**Sharing the host's kernel — the punchline.** A container does **not** boot its own Linux. There is no kernel inside it; every syscall goes straight to the host kernel. A VM, by contrast, virtualises the **hardware**, so each guest ships a full guest OS and its own kernel.

```
       Virtual Machines                    Containers
   +-----+ +-----+ +-----+            +-----+ +-----+ +-----+
   | App | | App | | App |            | App | | App | | App |
   +-----+ +-----+ +-----+            +-----+ +-----+ +-----+
   |Guest| |Guest| |Guest|            +---------------------+
   | OS  | | OS  | | OS  |            |   Docker Engine     |
   +-----+-+-----+-+-----+            +---------------------+
   |     Hypervisor      |            |  Host Linux kernel  |
   +---------------------+            +---------------------+
   |  Host OS / hardware |            |      Hardware       |
   +---------------------+            +---------------------+
```

A VM's isolation is *hypervisor-strong*; a container's is *kernel-strong* — lighter and faster (milliseconds to start, megabytes on disk), but sharing one kernel.

**The kernel features under the hood.** Docker doesn't invent isolation; it composes Linux primitives that already existed:

- **Namespaces** — give each container its own view of PIDs, mounts, network, users, hostname, and IPC.
- **Control groups (cgroups)** — meter and cap CPU, memory, I/O, and process count.
- **Capabilities** — split all-powerful root into ~40 fine-grained privileges, so a container's root isn't really *your* root.
- **Seccomp, AppArmor, SELinux** — restrict which syscalls and operations are allowed.

We meet each properly in later modules. For now: **Docker is a friendly front end on kernel primitives that were already there.**